# **Predicción del Sistema Operativo de un Usuario a Partir de su Comportamiento Digital**

Este dataset registra el comportamiento de usuarios en una plataforma digital, donde cada fila representa una sesión de navegación. El objetivo es predecir qué sistema operativo usa el usuario

## Limpieza de datos  

In [13]:
%pip install pandas==1.5.3
%pip install numpy==1.23.5
%pip install seaborn==0.12.2
%pip install matplotlib==3.7.0
%pip install scikit-learn==1.2.1
%pip install missingno==0.5.2

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [14]:
# Función imputación de outlier
# ------

def imputar_valores_extremos(df, variable, metodo='media'):
    """
    Imputa valores extremos en una variable de un DataFrame utilizando la media o la mediana.

    Parámetros:
    df (DataFrame): El DataFrame que contiene la variable a imputar.
    variable (str): El nombre de la variable que deseas imputar.
    metodo (str): La forma de imputación ('media' o 'mediana'). Por defecto es 'media'.

    Retorna:
    DataFrame: El DataFrame con la variable imputada.
    """
    if metodo not in ['media', 'mediana']:
        raise ValueError("El método debe ser 'media' o 'mediana'")

    # Calcular la media o la mediana
    if metodo == 'media':
        valor_imputacion = df[variable].mean()
    else:
        valor_imputacion = df[variable].median()

    # Identificar valores extremos (usando una regla de 3 veces la desviación estándar)
    limite_inferior = df[variable].mean() - 1.9 * df[variable].std()
    limite_superior = df[variable].mean() + 1.9 * df[variable].std()

    # Imputar valores extremos
    df[variable] = np.where(
        (df[variable] < limite_inferior) | (df[variable] > limite_superior),
        valor_imputacion,
        df[variable]
    )

    return df


# Función imputación perdidos
# ------

def imputar_valores(df, variable, metodo='media', valor_especifico=None):
    """
    Imputa valores perdidos en una columna de un DataFrame según el método especificado.

    Parámetros:
    df (pd.DataFrame): El DataFrame en el que se imputarán los valores.
    variable (str): El nombre de la columna a imputar.
    metodo (str): El método de imputación ('media', 'mediana', 'moda', 'valor_especifico').
    valor_especifico: El valor específico a usar para la imputación (relevante solo si 'metodo' es 'valor_especifico').

    Retorna:
    pd.DataFrame: El DataFrame con la variable imputada.
    """

    if metodo == 'media':
        imputacion = df[variable].mean()
    elif metodo == 'mediana':
        imputacion = df[variable].median()
    elif metodo == 'moda':
        imputacion = df[variable].mode()[0]
    elif metodo == 'valor_especifico':
        if valor_especifico is None:
            raise ValueError("Debe proporcionar un valor específico para la imputación.")
        imputacion = valor_especifico
    else:
        raise ValueError("Método de imputación no reconocido. Use 'media', 'mediana', 'moda' o 'valor_especifico'.")

    df[variable].fillna(imputacion, inplace=True)
    return df


def calcular_valores_perdidos(df):
    """
    Calcula la cantidad y el porcentaje de valores NaN en cada variable del DataFrame.

    Parámetros:
    df (pd.DataFrame): DataFrame con las variables a analizar.

    Retorna:
    pd.DataFrame: Un DataFrame con el resumen de valores perdidos, mostrando:
        - Variable: Nombre de la columna analizada.
        - Total: Cantidad total de filas en la variable.
        - Valores Perdidos: Cantidad de valores NaN en la variable.
        - % Valores Perdidos: Porcentaje de valores NaN respecto al total (con 2 decimales).
    """
    resumen_perdidos = []  # Lista para almacenar los resultados

    for col in df.columns:  # Analizar todas las columnas
        total = df[col].shape[0]  # Total de filas en la columna
        valores_perdidos = df[col].isna().sum()  # Conteo de valores NaN

        # Calcular el porcentaje de valores perdidos
        porcentaje_perdidos = (valores_perdidos / total) * 100 if total > 0 else 0

        # Agregar la información a la lista de resultados
        resumen_perdidos.append([col, total, valores_perdidos, f"{porcentaje_perdidos:.2f}%"])

    # Convertir la lista en un DataFrame
    df_resumen = pd.DataFrame(resumen_perdidos, columns=['Variable', 'Total', 'Valores Perdidos', '% Valores Perdidos'])

    return df_resumen



# Funcion graficadora confusion_marix
# ---
def confusion_matrix_graph(cm):
  plt.figure(figsize=(8, 6))
  sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
              xticklabels=['No', 'Yes'],
              yticklabels=['No', 'Yes'])
  plt.title('Matriz de Confusión')
  plt.xlabel('Predicción')
  plt.ylabel('Realidad')
  plt.show()


  # Función para detectar valores atípicos con el método IQR
def detectar_outliers(df):
    resumen = {}
    for col in df.select_dtypes(include=['number']):  # Solo columnas numéricas
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
        porcentaje_outliers = (len(outliers) / len(df)) * 100

        resumen[col] = f"{porcentaje_outliers:.1f}% ({len(outliers)} valores atípicos)"

    return resumen


In [ ]:
#Importamos las librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc
from collections import Counter
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.model_selection import train_test_split

data= pd.read_csv('../data/raw/usuarios_win_mac_lin_exp_cmp.csv', encoding='latin1', sep = ';')
data.head()


# Lista de variables numéricas
numeric_vars = data.select_dtypes(include=['number']).columns.tolist()

# Lista de variables categóricas
categorical_vars = data.select_dtypes(include=['object', 'category']).columns.tolist()


# Imputando variables con presencia de valores atípicos
# ----

data['acciones_imput'] = data['acciones']

# Imputar valores extremos en la columna 'variable1' usando la media
data = imputar_valores_extremos(data, 'acciones_imput', metodo='mediana')



# Aplicar la función al DataFrame
outliers_resumen = detectar_outliers(data)

# Imputando variables con presencia de valores atípicos
# ----

data['acciones_imput'] = data['acciones']

# Imputar valores extremos en la columna 'variable1' usando la media
data = imputar_valores_extremos(data, 'acciones_imput', metodo='mediana')


numeric_vars.remove('acciones')
numeric_vars.append('acciones_imput')
numeric_vars


data['hora_dia_imput'] = data['hora_dia']
data = imputar_valores(data,'hora_dia_imput',metodo='moda')
calcular_valores_perdidos(data)

categorical_vars.remove('hora_dia')
categorical_vars.append('hora_dia_imput')

data['duracion_imput'] = data['duracion'].fillna(data['duracion'].mean()) # Imputando los valores perdidos por la media
data['duracion_imput_2'] = data['duracion'].fillna(data['duracion'].median()) # Imputando los valores perdidos por la mediana
data['duracion_imput_3'] = data['duracion'].fillna(data['duracion'].mode()[0]) # Imputando los valores perdidos por la moda

calcular_valores_perdidos(data)


# Selección de variables para la imputación
vars_imputacion = ['duracion', 'paginas', 'acciones_imput', 'nivel_usuario', 'valor']

df_imputacion = data[vars_imputacion] # Filtrar solo las columnas necesarias
imputer = IterativeImputer(random_state=42) # Crear el imputador con un modelo base (ejemplo: regresión bayesiana)

# Aplicar la imputación
df_imputado = imputer.fit_transform(df_imputacion)
df_imputado = pd.DataFrame(df_imputado, columns=vars_imputacion)
data['duracion_imput_4'] = df_imputado['duracion']

numeric_vars.remove('duracion')
numeric_vars.append('duracion_imput_4')

numeric_vars.remove('clase')

# Guardar todas las variables categoricas en un solo lugar
cat_cols = data[categorical_vars]
num_cols = data[numeric_vars]

# Generar variables para las dos columnas que omiti de mi mapeo de variables cualitativas y cuantitativas
label = data["clase"]

cat_cols = pd.get_dummies(data = cat_cols) #transformamos las variables categóricas a numéricas

df_complete = pd.concat([num_cols, cat_cols, label], axis=1)

df_complete.head()

,paginas,valor,edad,nivel_usuario,acciones_imput,duracion_imput_4,pais_Argentina,pais_Chile,pais_España,pais_Mexico,...,experiencia_baja,experiencia_media,navegador_Chrome,navegador_Edge,navegador_Firefox,navegador_Safari,hora_dia_imput_mañana,hora_dia_imput_noche,hora_dia_imput_tarde,clase
0,2,8,25.645123,4.394008,4.0,7.0,0,1,0,0,...,0,1,0,0,1,0,0,1,0,2
1,2,6,55.488254,2.887318,6.0,21.0,0,0,1,0,...,0,0,1,0,0,0,0,0,1,2
2,2,4,30.551096,2.609819,4.0,57.0,1,0,0,0,...,0,0,0,0,0,1,1,0,0,2
3,3,12,37.965528,2.214545,6.0,101.0,0,0,0,0,...,0,1,0,0,1,0,0,1,0,2
4,2,12,57.256667,3.040157,6.0,109.0,0,0,1,0,...,0,0,1,0,0,0,0,0,1,2


In [16]:
df_complete.shape

(1230, 22)

## Particionado de datos 

In [ ]:
X = df_complete.drop("clase",axis=1) # covariables
y = df_complete['clase'] # target
X_train_res, X_test, y_train_res, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE
from collections import Counter

In [ ]:
oversampler = RandomOverSampler(random_state=42)

X_train_over, y_train_over = oversampler.fit_resample(X_train_res, y_train_res)

X_train = X_train_over
y_train = y_train_over


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
# from xgboost import XGBClassifier
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier

from sklearn.multiclass import OneVsRestClassifier

#### Decision Tree - Multiclasss
One-vs-Rest (OvR)